# AI Crop Disease Detector — full training pipeline (Google Colab, 61 classes)

**Runtime (Colab Pro):** Runtime -> Change runtime type ->
  *  **L4 GPU** (best fit for MobileNetV2; pick **A100 GPU** if L4 quota is out)  
  *  **High-RAM: ON**  
  *  Runtime version: **Latest (recommended)**

Every number this notebook prints is real — copy outputs into ml/reports/, never edit them.

Steps in this notebook:
  1. clone the repo, install deps
  2. copy the pre-made split from Google Drive (data/processed — 97,480 images, 61 classes)
  3. verify the split (61 classes, no empty class)
  4. `train.py` — two-stage transfer learning (the long cell: ~3-5 h)
  5. `evaluate.py` + `calibrate_threshold.py` + `export_onnx.py`
  6. zip the artifacts -> download -> send back to the repo owner


## 1. Clone repo + install dependencies (~3 min)

In [ ]:
# 1a. Clone the repository and cd into it.
# `|| true`: re-running the notebook must not fail when the folder
# already exists (the 15 GB of processed images live inside it).
!git clone https://github.com/ibrahimSumon47/AI-Crop-Disease-Detector-for-Bangladeshi-Farmers.git || true
%cd /content/AI-Crop-Disease-Detector-for-Bangladeshi-Farmers
!git log --oneline -3

In [ ]:
# 1b. Install the pinned ML dependencies from ml/requirements.txt.
# Colab already ships a recent TensorFlow; the requirements file only adds
# what is missing (tf2onnx, onnx/onnxruntime, etc.). --no-deps-free quick install.
!pip install -q onnx tf2onnx 'onnxruntime>=1.16'
!pip install -q -r ml/requirements.txt
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'NO GPU! Runtime -> Change runtime type -> T4 GPU, then re-run this cell.'

## 1c. কোড ফিক্স — ৪টা `.py` ফাইল ঠিক করে লেখা (এই cell-গুলো আগে রান করুন)

গত রানে যে ৩টা error এসেছিল সেগুলোর সমাধান এখানে বেক করা আছে, তাই git-এ
পুশ হওয়ার আগেও আপনি ট্রেনিং চালাতে পারবেন:

1. `augmentations.py` — Keras-এর `RandomRotation`/`RandomZoom` বাদ; ওগুলো
   `tf.data.map()`-এর ভেতরে `tf.Variable` বানিয়ে
   `singletTFVariable` error দিচ্ছিল। এখন stateless TF ops ব্যবহার হচ্ছে।
2. `classes.py` — `missing_from_dataset()` যোগ; ডেটাসেটে ১১টা ক্লাস থাকলেও
   (ধানের ব্যাকটেরিয়াল লিফ ব্লাইট + টুংরো নেই) ট্রেনিং চলবে, আর ভুল নামের
   ফোল্ডার হলে তখনও থেমে যাবে।
3. `train.py` — "folder order differs from classes.py" চেক বাদ। Keras ক্লাস
   নম্বর দেয় বর্ণানুক্রমিক folder নামে, তাই order কখনোই মিলত না। এখন আসল
   order `ml/reports/class_order.json`-এ লেখা হয়, হেডের সাইজ
   `len(class_names)`, class weights-ও সেই order-এ।
4. `export_onnx.py` — `labels.json` এখন মডেলের আসল output order-এ লেখা হয়
   (আগে ভুল order-এ যেত, ফলে প্রতিটি প্রেডিকশন ভুল রোগে ম্যাপ হতো)।

নিচের ৪টা `%%writefile` cell চালালেই ফাইলগুলো `ml/src/`-তে বসে যাবে।
এগুলো চালানো বাধ্যতামূলক — ট্রেনিং cell-এর **আগে**।

In [ ]:
%%writefile ml/src/augmentations.py
"""
augmentations.py — training-time image augmentation pipeline.

WHY AUGMENT
    Farm photos look very different from the lab datasets: different light,
    different backgrounds (soil, sky, a hand holding the leaf), camera shake,
    zoom level. Augmentation teaches the network to ignore those factors and
    focus on the leaf symptoms themselves. It is the single biggest defence
    against the lab-vs-field accuracy gap.

WHAT IT PRODUCES
    `build_datasets(train_dir, val_dir)` returns tf.data pipelines that resize
    and, for TRAINING ONLY, randomly apply per image:
      * random rotation (±8.6°)   — leaves point in any direction
      * random horizontal flip    — left/right symmetry of leaves
      * random vertical flip      — plants photographed from any angle
      * brightness jitter (±20 %) — early morning vs noon light
      * contrast jitter (±20 %)   — hazy vs sunny conditions
      * random zoom/crop (0.8–1.0× crop) — arm's length vs close-up photos
      * mild saturation/hue shift — different phone camera colour science
      * slight blur (10 % of images) — cheap-phone motion shake

    Validation/test data get ONLY resize + normalize, so reported metrics stay
    trustworthy (never augment the data you measure on).

IMPORTANT IMPLEMENTATION NOTE (why no tf.keras.layers here)
    Keras preprocessing layers such as RandomRotation/RandomZoom create an
    internal seed-generator variable the first time they are called. Calling
    them inside a `tf.data.Dataset.map()` function therefore raises
        ValueError: tf.function-decorated function tried to create variables
        on non-first call  ("singletTFVariable")
    as soon as tf.data re-traces the function for a second worker thread.
    That exact crash killed our first Colab run, so every transform below is
    written with STATELESS TensorFlow ops (tf.image.*, tf.raw_ops.*) instead.
    Same randomness, no variables, safe under num_parallel_calls=AUTOTUNE.

SERVING CONTRACT
    The constants here (input size, normalization scheme) are the contract
    with the backend: backend/app/services/preprocess.py must reproduce
    exactly this resize + [-1, +1] scaling, otherwise live predictions drift
    even though validation accuracy looks fine.
"""

from __future__ import annotations

import tensorflow as tf

# ---------------------------------------------------------------------------
# Constants — single source of truth for training AND serving preprocessing.
# ---------------------------------------------------------------------------
IMAGE_SIZE = (224, 224)  # MobileNetV2 native input resolution
NORMALIZATION = "mobilenet_v2"  # scale pixels to [-1, +1]

# Augmentation strengths: deliberately moderate. Too strong and the model
# learns noise; too weak and it simply memorises the lab look.
ROTATION_RANGE = 0.15  # radians, ~ ±8.6 degrees
BRIGHTNESS_JITTER = 0.2  # ±20 %
CONTRAST_JITTER = 0.2  # ±20 %
ZOOM_MIN_CROP = 0.80  # keep 80–100 % of each side, then resize back up
BLUR_PROBABILITY = 0.10  # 10 % of training images get a light blur
HUE_JITTER = 0.02  # keep hue shifts tiny: colour is a symptom signal!
SATURATION_MIN, SATURATION_MAX = 0.85, 1.15


# ---------------------------------------------------------------------------
# Individual ops (tiny readable wrappers, all stateless)
# ---------------------------------------------------------------------------
def resize_image(image: tf.Tensor) -> tf.Tensor:
    """Resize any input image to the fixed model input shape, keeping RGB."""
    return tf.image.resize(tf.convert_to_tensor(image), IMAGE_SIZE)


def random_rotation(image: tf.Tensor) -> tf.Tensor:
    """Rotate by a random angle in [-ROTATION_RANGE, +ROTATION_RANGE] radians.

    Implemented with `tf.raw_ops.ImageProjectiveTransformV3` (GPU-friendly,
    stateless). The 2x3 matrix maps output pixel coordinates back to input
    coordinates, so we build the INVERSE rotation matrix:

        [  cos  sin  0 ]
        [ -sin  cos  0 ]

    Out-of-frame pixels are filled by reflection so no black corners appear.
    """
    angle = tf.random.uniform([], -ROTATION_RANGE, ROTATION_RANGE)
    cos_a = tf.cos(angle)
    sin_a = tf.sin(angle)

    # transforms must be a [batch, 8] float32 tensor (the op reads the first
    # six entries of the 2x3 affine matrix; the last two are padding).
    transform = tf.stack(
        [cos_a, sin_a, 0.0, -sin_a, cos_a, 0.0, 0.0, 0.0], axis=0
    )
    batched = tf.expand_dims(tf.cast(image, tf.float32), axis=0)
    rotated = tf.raw_ops.ImageProjectiveTransformV3(
        images=batched,
        transforms=tf.expand_dims(transform, axis=0),
        output_shape=tf.constant(IMAGE_SIZE, dtype=tf.int32),
        fill_value=tf.constant(0.0, dtype=tf.float32),
        interpolation="BILINEAR",
        fill_mode="REFLECT",
    )
    return tf.squeeze(rotated, axis=0)


def random_flips(image: tf.Tensor) -> tf.Tensor:
    """Random horizontal + vertical flip (leaves are symmetric enough for this)."""
    image = tf.image.random_flip_left_right(image)
    return tf.image.random_flip_up_down(image)


def random_brightness(image: tf.Tensor) -> tf.Tensor:
    """Brightness jitter — early-morning vs midday sunlight."""
    return tf.image.random_brightness(image, max_delta=BRIGHTNESS_JITTER)


def random_contrast(image: tf.Tensor) -> tf.Tensor:
    """Contrast jitter — hazy monsoon air vs hard direct sun."""
    return tf.image.random_contrast(
        image, lower=1.0 - CONTRAST_JITTER, upper=1.0 + CONTRAST_JITTER
    )


def random_zoom(image: tf.Tensor) -> tf.Tensor:
    """Zoom by cropping 80–100 % of the image and resizing back to 224x224.

    Pure tf.image ops (no Keras layer, no variables):
      1. pick a random crop fraction in [ZOOM_MIN_CROP, 1.0]
      2. crop that window at a random offset
      3. resize it back to the model input size -> a zoom-in effect
    """
    crop_fraction = tf.random.uniform([], ZOOM_MIN_CROP, 1.0)
    height = tf.shape(image)[0]
    width = tf.shape(image)[1]

    new_h = tf.cast(tf.cast(height, tf.float32) * crop_fraction, tf.int32)
    new_w = tf.cast(tf.cast(width, tf.float32) * crop_fraction, tf.int32)
    # Always keep at least 1 pixel so crop_to_bounding_box never fails.
    new_h = tf.maximum(new_h, 1)
    new_w = tf.maximum(new_w, 1)

    # Random (not necessarily centred) offset => also acts as a random crop.
    max_top = tf.maximum(height - new_h, 0)
    max_left = tf.maximum(width - new_w, 0)
    top = tf.random.uniform([], 0, max_top + 1, dtype=tf.int32)
    left = tf.random.uniform([], 0, max_left + 1, dtype=tf.int32)

    cropped = tf.image.crop_to_bounding_box(image, top, left, new_h, new_w)
    return tf.image.resize(cropped, IMAGE_SIZE)


def random_color_jitter(image: tf.Tensor) -> tf.Tensor:
    """Mild saturation / hue shift — different phones render colour differently.

    Kept intentionally small: yellowing, browning and lesion colour are real
    diagnostic signals, so heavy colour jitter would destroy information.
    """
    image = tf.image.random_saturation(
        image, lower=SATURATION_MIN, upper=SATURATION_MAX
    )
    return tf.image.random_hue(image, max_delta=HUE_JITTER)


def maybe_blur(image: tf.Tensor) -> tf.Tensor:
    """With probability BLUR_PROBABILITY, apply a light 3x3 Gaussian-ish blur.

    Simulates cheap-phone motion shake / slightly out-of-focus shots.

    Kernel shape matters: `tf.nn.depthwise_conv2d` expects
    [kH, kW, in_channels, channel_multiplier]. For a 3-channel RGB image with
    one output channel per input channel that is [3, 3, 3, 1] — an earlier
    version passed [3, 3, 1, 1], which made the op demand a 1-channel input
    and crashed the whole Colab training run.
    """
    # 3x3 Gaussian-like kernel (sums to 1), applied independently per channel.
    kernel_2d = tf.constant(
        [[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]], dtype=tf.float32
    ) / 16.0
    kernel = tf.tile(tf.reshape(kernel_2d, [3, 3, 1, 1]), [1, 1, 3, 1])

    def _blur() -> tf.Tensor:
        batched = tf.expand_dims(image, axis=0)
        blurred = tf.nn.depthwise_conv2d(
            batched, kernel, strides=[1, 1, 1, 1], padding="SAME"
        )
        return tf.squeeze(blurred, axis=0)

    should_blur = tf.random.uniform([], 0.0, 1.0) < BLUR_PROBABILITY
    # tf.cond keeps the graph pure and works inside tf.data pipelines.
    return tf.cond(should_blur, _blur, lambda: image)


def normalize_image(image: tf.Tensor) -> tf.Tensor:
    """Scale pixel values from [0, 255] to [-1, +1] (MobileNetV2 convention)."""
    image = tf.cast(image, tf.float32)
    return (image / 127.5) - 1.0


# ---------------------------------------------------------------------------
# Pipeline builders
# ---------------------------------------------------------------------------
def train_transform(image: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    """Full augmentation for TRAIN data: resize -> augment (in [0,1]) -> [-1,1]."""
    image = resize_image(image)
    # Augment in the [0, 1] range: tf.image ops expect/return that range.
    image = tf.cast(image, tf.float32) / 255.0

    image = random_flips(image)
    image = random_rotation(image)
    image = random_brightness(image)
    image = random_contrast(image)
    image = random_zoom(image)
    image = random_color_jitter(image)
    image = maybe_blur(image)

    # Clamp before scaling up so brightness/contrast do not overshoot.
    image = tf.clip_by_value(image, 0.0, 1.0) * 255.0
    image = normalize_image(image)
    return image, label


def eval_transform(image: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    """NO augmentation for val/test — only resize + normalize (deterministic metrics)."""
    image = resize_image(image)
    return normalize_image(image), label


def build_datasets(
    train_dir: str,
    val_dir: str,
    batch_size: int = 32,
    seed: int = 42,
) -> tuple[tf.data.Dataset, tf.data.Dataset, list[str]]:
    """Build tf.data pipelines straight from the split folders.

    Args:
        train_dir: folder of `train/<class_name>/*.jpg` images.
        val_dir:   folder of `val/<class_name>/*.jpg` images.
        batch_size: images per batch (32 fits a free Colab/Kaggle T4).
        seed: fixed seed so shuffling is reproducible across runs.

    Returns:
        (train_ds, val_ds, class_names) where `class_names` is the alphabetical
        folder order used by Keras. That SAME order must be written to
        labels.json by export_onnx.py, otherwise the index <-> disease mapping
        in the backend would silently be wrong.
    """
    common = dict(
        labels="inferred",
        label_mode="int",
        image_size=IMAGE_SIZE,
        batch_size=batch_size,
    )
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, seed=seed, **common
    )
    # shuffle=False for validation so metrics are stable between epochs.
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, seed=seed, shuffle=False, **common
    )

    class_names = list(train_ds.class_names)

    # Augment only the training split, then batch and prefetch for GPU speed.
    train_ds = (
        train_ds.unbatch()
        .map(train_transform, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        val_ds.unbatch()
        .map(eval_transform, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    return train_ds, val_ds, class_names


In [ ]:
%%writefile ml/src/classes.py
"""
classes.py — the single source of truth for the model's class list.

Why one canonical file?
    Every other component (data prep script, training script, backend
    labels.json, backend inference, frontend display) must agree on the
    class ORDER. A mismatch silently shuffles predictions — e.g. the model
    says "tomato late blight" but the backend reads it as "potato healthy".
    So there is exactly one place that defines the order, and
    `export_onnx.py` writes `labels.json` straight from this file.

Class naming rules
    - Lowercase snake_case, prefixed with the crop: `<crop>_<disease>`.
    - Every crop has its own `healthy` class, so the model learns crop +
      disease together in a single multi-class head.

Adding a new crop (v2 expansion)
    The original 13 classes were rice / potato / tomato. Canon has since grown
    with 8 more crops sourced from free public datasets — the canonical class
    list below is the FULL target. A trained model only covers the subset its
    training data actually contained (train.py supports subsets and
    export_onnx.py writes labels.json from what was really trained), so this
    file may be ahead of the deployed model at any time. Eggplant remains a
    TODO: OLID I's class names must be confirmed before adding those classes.
"""

from __future__ import annotations

# ---------------------------------------------------------------------------
# CLASSES — the final, ordered list. Index in this list == model output index.
#
# Selected to match what the three public datasets actually contain:
#   * PlantVillage provides potato (early/late blight + healthy) and tomato
#     (many diseases; we take a practical subset).
#   * A public rice leaf disease dataset (e.g. Rice Leaf Diseases from
#     UCI/Mendeley) provides brown_spot, blast, bacterial_leaf_blight, tungro.
# If a chosen dataset turns out to lack a class, this list must be revised
# TOGETHER WITH the user before training (see T2.2 notes).
# ---------------------------------------------------------------------------
CLASSES: list[str] = [
    # --- rice (ধান) ---
    "rice_bacterial_leaf_blight",
    "rice_brown_spot",
    "rice_blast",
    "rice_tungro",
    "rice_healthy",
    # --- potato (আলু) ---
    "potato_early_blight",
    "potato_late_blight",
    "potato_healthy",
    # --- tomato (টম্যাটো) ---
    "tomato_bacterial_spot",
    "tomato_early_blight",
    "tomato_late_blight",
    "tomato_leaf_mold",
    "tomato_healthy",
    # --- chilli (মরিচ) — Bangladesh chilli dataset (Mendeley w9mr3vf56s) ---
    "chilli_bacterial_spot",
    "chilli_curl_virus",
    "chilli_cercospora_leaf_spot",
    "chilli_nutrient_deficiency",
    "chilli_white_spot",
    "chilli_healthy",
    # --- maize / corn (ভুট্টা) — PlantVillage + Tanzania maize (Zenodo) ---
    "maize_common_rust",
    "maize_gray_leaf_spot",
    "maize_northern_leaf_blight",
    "maize_lethal_necrosis",
    "maize_streak_virus",
    "maize_healthy",
    # --- grape (আঙুর) — PlantVillage + NGLD (Mendeley 8nnd2ypcv3) ---
    "grape_black_rot",
    "grape_esca",
    "grape_leaf_blight",
    "grape_downy_mildew",
    "grape_powdery_mildew",
    "grape_bacterial_leaf_spot",
    "grape_healthy",
    # --- cauliflower (ফুলকপি) — BD cauliflower dataset (Mendeley x995snz7p3) ---
    "cauliflower_black_rot",
    "cauliflower_insect_holes",
    "cauliflower_healthy",
    # --- cucumber (শসা) — Cucumber Disease Recognition (Mendeley y6d3z6f8z9) ---
    "cucumber_anthracnose",
    "cucumber_bacterial_wilt",
    "cucumber_belly_rot",
    "cucumber_downy_mildew",
    "cucumber_pythium_fruit_rot",
    "cucumber_gummy_stem_blight",
    "cucumber_healthy",
    # --- papaya (পেঁপে) — BDPapayaLeaf (Mendeley p997fvf526) ---
    "papaya_anthracnose",
    "papaya_bacterial_spot",
    "papaya_curl",
    "papaya_ring_spot",
    "papaya_healthy",
    # --- mango (আম) — MangoLeafBD (Mendeley hxsnvwty3r) ---
    "mango_anthracnose",
    "mango_bacterial_canker",
    "mango_cutting_weevil",
    "mango_die_back",
    "mango_gall_midge",
    "mango_powdery_mildew",
    "mango_sooty_mould",
    "mango_healthy",
    # --- okra (ঢেঁড়স) — Okra DiseaseNet (Mendeley nh7zk4hv8z) ---
    "okra_alternaria_leaf_spot",
    "okra_cercospora_leaf_spot",
    "okra_downy_mildew",
    "okra_leaf_curl_virus",
    "okra_phyllosticta_leaf_spot",
    "okra_healthy",
]

# ---------------------------------------------------------------------------
# Derived mappings (always derived from CLASSES — never edited by hand).
# ---------------------------------------------------------------------------
N_CLASSES: int = len(CLASSES)

# Model output index -> class label
INDEX_TO_LABEL: dict[int, str] = {i: name for i, name in enumerate(CLASSES)}

# Class label -> model output index
LABEL_TO_INDEX: dict[str, int] = {name: i for i, name in enumerate(CLASSES)}


def crop_of(label: str) -> str:
    """Split '<crop>_<disease>' into the crop part (before the first '_')."""
    return label.split("_", 1)[0]


def disease_of(label: str) -> str:
    """Split '<crop>_<disease>' into the disease part (else 'healthy')."""
    return label.split("_", 1)[1]


# ---------------------------------------------------------------------------
# Human-readable names — used by the backend labels.json and the guideline
# lookup so the UI can show "পাতার ব্লাস্ট" instead of "rice_blast".
# ---------------------------------------------------------------------------
EN_NAMES: dict[str, str] = {
    "rice_bacterial_leaf_blight": "Rice Bacterial Leaf Blight",
    "rice_brown_spot": "Rice Brown Spot",
    "rice_blast": "Rice Blast",
    "rice_tungro": "Rice Tungro",
    "rice_healthy": "Rice - Healthy",
    "potato_early_blight": "Potato Early Blight",
    "potato_late_blight": "Potato Late Blight",
    "potato_healthy": "Potato - Healthy",
    "tomato_bacterial_spot": "Tomato Bacterial Spot",
    "tomato_early_blight": "Tomato Early Blight",
    "tomato_late_blight": "Tomato Late Blight",
    "tomato_leaf_mold": "Tomato Leaf Mold",
    "tomato_healthy": "Tomato - Healthy",
    # --- chilli ---
    "chilli_bacterial_spot": "Chilli Bacterial Spot",
    "chilli_curl_virus": "Chilli Curl Virus",
    "chilli_cercospora_leaf_spot": "Chilli Cercospora Leaf Spot",
    "chilli_nutrient_deficiency": "Chilli Nutrient Deficiency",
    "chilli_white_spot": "Chilli White Spot",
    "chilli_healthy": "Chilli - Healthy",
    # --- maize ---
    "maize_common_rust": "Maize Common Rust",
    "maize_gray_leaf_spot": "Maize Gray Leaf Spot",
    "maize_northern_leaf_blight": "Maize Northern Leaf Blight",
    "maize_lethal_necrosis": "Maize Lethal Necrosis",
    "maize_streak_virus": "Maize Streak Virus",
    "maize_healthy": "Maize - Healthy",
    # --- grape ---
    "grape_black_rot": "Grape Black Rot",
    "grape_esca": "Grape Esca",
    "grape_leaf_blight": "Grape Leaf Blight",
    "grape_downy_mildew": "Grape Downy Mildew",
    "grape_powdery_mildew": "Grape Powdery Mildew",
    "grape_bacterial_leaf_spot": "Grape Bacterial Leaf Spot",
    "grape_healthy": "Grape - Healthy",
    # --- cauliflower ---
    "cauliflower_black_rot": "Cauliflower Black Rot",
    "cauliflower_insect_holes": "Cauliflower Insect Holes",
    "cauliflower_healthy": "Cauliflower - Healthy",
    # --- cucumber ---
    "cucumber_anthracnose": "Cucumber Anthracnose",
    "cucumber_bacterial_wilt": "Cucumber Bacterial Wilt",
    "cucumber_belly_rot": "Cucumber Belly Rot",
    "cucumber_downy_mildew": "Cucumber Downy Mildew",
    "cucumber_pythium_fruit_rot": "Cucumber Pythium Fruit Rot",
    "cucumber_gummy_stem_blight": "Cucumber Gummy Stem Blight",
    "cucumber_healthy": "Cucumber - Healthy",
    # --- papaya ---
    "papaya_anthracnose": "Papaya Anthracnose",
    "papaya_bacterial_spot": "Papaya Bacterial Spot",
    "papaya_curl": "Papaya Curl",
    "papaya_ring_spot": "Papaya Ring Spot",
    "papaya_healthy": "Papaya - Healthy",
    # --- mango ---
    "mango_anthracnose": "Mango Anthracnose",
    "mango_bacterial_canker": "Mango Bacterial Canker",
    "mango_cutting_weevil": "Mango Cutting Weevil",
    "mango_die_back": "Mango Die Back",
    "mango_gall_midge": "Mango Gall Midge",
    "mango_powdery_mildew": "Mango Powdery Mildew",
    "mango_sooty_mould": "Mango Sooty Mould",
    "mango_healthy": "Mango - Healthy",
    # --- okra ---
    "okra_alternaria_leaf_spot": "Okra Alternaria Leaf Spot",
    "okra_cercospora_leaf_spot": "Okra Cercospora Leaf Spot",
    "okra_downy_mildew": "Okra Downy Mildew",
    "okra_leaf_curl_virus": "Okra Leaf Curl Virus",
    "okra_phyllosticta_leaf_spot": "Okra Phyllosticta Leaf Spot",
    "okra_healthy": "Okra - Healthy",
}

# Bangla names — served in the prediction response for the Bangla-first UI.
BN_NAMES: dict[str, str] = {
    "rice_bacterial_leaf_blight": "ধান — ব্যাকটেরিয়াল লিফ ব্লাইট",
    "rice_brown_spot": "ধান — বাদামি দাগ",
    "rice_blast": "ধান — ব্লাস্ট রোগ",
    "rice_tungro": "ধান — টুংরো রোগ",
    "rice_healthy": "ধান — সুস্থ পাতা",
    "potato_early_blight": "আলু — আগাম পাতা পোড়া (Early Blight)",
    "potato_late_blight": "আলু — অনাগাম পাতা পোড়া (Late Blight)",
    "potato_healthy": "আলু — সুস্থ পাতা",
    "tomato_bacterial_spot": "টম্যাটো — ব্যাকটেরিয়াল স্পট",
    "tomato_early_blight": "টম্যাটো — আগাম পাতা পোড়া (Early Blight)",
    "tomato_late_blight": "টম্যাটো — অনাগাম পাতা পোড়া (Late Blight)",
    "tomato_leaf_mold": "টম্যাটো — লিফ মোল্ড",
    "tomato_healthy": "টম্যাটো — সুস্থ পাতা",
    # --- chilli (মরিচ) ---
    "chilli_bacterial_spot": "মরিচ — ব্যাকটেরিয়াল স্পট",
    "chilli_curl_virus": "মরিচ — পাতা কোঁকড়ানো ভাইরাস",
    "chilli_cercospora_leaf_spot": "মরিচ — সারকোস্পোরা পাতা দাগ",
    "chilli_nutrient_deficiency": "মরিচ — পুষ্টি উপাদানের ঘাটতি",
    "chilli_white_spot": "মরিচ — সাদা দাগ",
    "chilli_healthy": "মরিচ — সুস্থ পাতা",
    # --- maize (ভুট্টা) ---
    "maize_common_rust": "ভুট্টা — সাধারণ মরিচা (Common Rust)",
    "maize_gray_leaf_spot": "ভুট্টা — ধূসর পাতা দাগ (Gray Leaf Spot)",
    "maize_northern_leaf_blight": "ভুট্টা — উত্তর পাতা পোড়া (Northern Leaf Blight)",
    "maize_lethal_necrosis": "ভুট্টা — লিথাল নেক্রোসিস (মারাত্মক পাতা মৃত্যু)",
    "maize_streak_virus": "ভুট্টা — স্ট্রিক ভাইরাস",
    "maize_healthy": "ভুট্টা — সুস্থ পাতা",
    # --- grape (আঙুর) ---
    "grape_black_rot": "আঙুর — কালো পচা (Black Rot)",
    "grape_esca": "আঙুর — এসকা (Esca)",
    "grape_leaf_blight": "আঙুর — পাতা পোড়া (Leaf Blight)",
    "grape_downy_mildew": "আঙুর — ডাউনি মিলডিউ",
    "grape_powdery_mildew": "আঙুর — পাউডারি মিলডিউ",
    "grape_bacterial_leaf_spot": "আঙুর — ব্যাকটেরিয়াল পাতা দাগ",
    "grape_healthy": "আঙুর — সুস্থ পাতা",
    # --- cauliflower (ফুলকপি) ---
    "cauliflower_black_rot": "ফুলকপি — কালো পচা (Black Rot)",
    "cauliflower_insect_holes": "ফুলকপি — পোকার ছিদ্র",
    "cauliflower_healthy": "ফুলকপি — সুস্থ পাতা",
    # --- cucumber (শসা) ---
    "cucumber_anthracnose": "শসা — অ্যানথ্রাকনোজ",
    "cucumber_bacterial_wilt": "শসা — ব্যাকটেরিয়াল উইল্ট",
    "cucumber_belly_rot": "শসা — বেলি রট (ফল পচা)",
    "cucumber_downy_mildew": "শসা — ডাউনি মিলডিউ",
    "cucumber_pythium_fruit_rot": "শসা — পাইথিয়াম ফল পচা",
    "cucumber_gummy_stem_blight": "শসা — গামি স্টেম ব্লাইট",
    "cucumber_healthy": "শসা — সুস্থ পাতা",
    # --- papaya (পেঁপে) ---
    "papaya_anthracnose": "পেঁপে — অ্যানথ্রাকনোজ",
    "papaya_bacterial_spot": "পেঁপে — ব্যাকটেরিয়াল স্পট",
    "papaya_curl": "পেঁপে — পাতা কোঁকড়ানো",
    "papaya_ring_spot": "পেঁপে — রিং স্পট ভাইরাস",
    "papaya_healthy": "পেঁপে — সুস্থ পাতা",
    # --- mango (আম) ---
    "mango_anthracnose": "আম — অ্যানথ্রাকনোজ",
    "mango_bacterial_canker": "আম — ব্যাকটেরিয়াল ক্যানসার",
    "mango_cutting_weevil": "আম — কাটিং উইভিল পোকা",
    "mango_die_back": "আম — ডাই-ব্যাক (ডগা শুকিয়ে যাওয়া)",
    "mango_gall_midge": "আম — গল মিজ (পাতা ফোলা)",
    "mango_powdery_mildew": "আম — পাউডারি মিলডিউ",
    "mango_sooty_mould": "আম — সুটি ছাঁচ (কালো আস্তরণ)",
    "mango_healthy": "আম — সুস্থ পাতা",
    # --- okra (ঢেঁড়স) ---
    "okra_alternaria_leaf_spot": "ঢেঁড়স — অলটারনেরিয়া পাতা দাগ",
    "okra_cercospora_leaf_spot": "ঢেঁড়স — সারকোস্পোরা পাতা দাগ",
    "okra_downy_mildew": "ঢেঁড়স — ডাউনি মিলডিউ",
    "okra_leaf_curl_virus": "ঢেঁড়স — পাতা কোঁকড়ানো ভাইরাস",
    "okra_phyllosticta_leaf_spot": "ঢেঁড়স — ফাইলোস্টিক্টা পাতা দাগ",
    "okra_healthy": "ঢেঁড়স — সুস্থ পাতা",
}


def missing_from_dataset(folders: list[str]) -> list[str]:
    """Validate a dataset's class folders against the canonical class list.

    IMPORTANT — what the model's label order actually is:
        Keras (`image_dataset_from_directory`) builds its label index from the
        ALPHABETICAL order of the folder names, so the trained model's output
        order is the DATASET order, not this file's order (this file groups by
        crop: rice, potato, tomato). train.py therefore records the real order
        in ml/reports/class_order.json and export_onnx.py replays that exact
        order when it writes labels.json. This function only guards the SET of
        names, which is what actually protects us from silent label swaps.

    Args:
        folders: class folder names found in data/processed/train.

    Returns:
        Canonical classes the dataset does NOT contain. We train on what the
        data actually has (subset support) and report the rest, instead of
        crashing — e.g. the public rice dataset we downloaded has blast,
        brown spot and healthy, but no bacterial leaf blight / tungro images.

    Raises:
        SystemExit: when a folder is not a known class. That means the data
        preparation step produced a typo'd folder name, which would otherwise
        turn into an unknown 12th output index.
    """
    unknown = sorted({f for f in folders if f not in LABEL_TO_INDEX})
    if unknown:
        raise SystemExit(
            "ERROR: unknown dataset class folder(s): "
            f"{unknown}\nKnown classes: {sorted(CLASSES)}\n"
            "Fix prepare_data.py aliases or rename the folders."
        )
    present = set(folders)
    return [c for c in CLASSES if c not in present]


def validate() -> None:
    """Self-check called by other scripts: all mappings must be consistent."""
    assert len(set(CLASSES)) == N_CLASSES, "duplicate class names!"
    assert INDEX_TO_LABEL[N_CLASSES - 1] == CLASSES[-1]
    assert sorted(INDEX_TO_LABEL) == list(range(N_CLASSES))
    assert set(EN_NAMES) == set(CLASSES), "EN_NAMES missing/extra entries"
    assert set(BN_NAMES) == set(CLASSES), "BN_NAMES missing/extra entries"
    assert set(crop_of(c) for c in CLASSES) == {
        "rice", "potato", "tomato", "chilli", "maize", "grape",
        "cauliflower", "cucumber", "papaya", "mango", "okra",
    }
    print(
        f"classes.py OK — {N_CLASSES} classes across "
        f"{len(set(crop_of(c) for c in CLASSES))} crops: "
        + ", ".join(sorted(set(crop_of(c) for c in CLASSES)))
    )


if __name__ == "__main__":
    validate()


In [ ]:
%%writefile ml/src/train.py
"""
train.py — two-stage transfer-learning training.

STAGE 1 (head training)
    Backbone frozen. Only the new Dense head learns. Few epochs, higher LR.
STAGE 2 (fine-tuning)
    Unfreeze the top `FINE_TUNE_LAYERS` backbone layers (model.py), tiny LR
    (STAGE2_LR = 1e-5) so pretrained weights stay useful.

Other pieces (per project spec):
    * class weights  — computed from the training split to counterbalance
                       the heavy PlantVillage tomato classes.
    * early stopping — on val_accuracy with patience; restores best weights.
    * checkpointing  — saves the best model by val_accuracy per stage.
    * history.json   — full Keras history + stage metadata saved to
                       ml/reports/, so the final report uses REAL numbers.

USAGE (GPU expected — Kaggle/Colab; CPU works but slow)
    python ml/src/train.py --train data/processed/train --val data/processed/val \
        [--epochs-head 5 --epochs-ft 15 --batch 32 --out-model ml/reports/best_model.keras]
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import tensorflow as tf  # module level: make_callbacks() needs it at call time

from augmentations import build_datasets
from classes import missing_from_dataset
from model import STAGE1_LR, STAGE2_LR, build_model, unfreeze_for_finetune

REPO_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_REPORT_DIR = REPO_ROOT / "ml" / "reports"

# ---------------------------------------------------------------------------
# Early stopping / patience settings
# ---------------------------------------------------------------------------
PATIENCE_HEAD = 3               # stop head training quickly if no progress
PATIENCE_FT = 5                 # fine-tuning is slower; allow more patience


def compute_class_weights(class_names: list[str], train_dir: Path) -> dict[int, float]:
    """Compute class weights from the number of images per class folder.

    Formula: n_samples / (n_classes * n_class_samples) — the sklearn
    'balanced' convention. Keeps rare classes (rice diseases) from being
    drowned out by tomato's ~18k images.

    Args:
        class_names: the model's real output order (the Keras folder order),
            NOT classes.py order — otherwise the weight for index 0 would be
            applied to the wrong class and quietly bias the model.
        train_dir: processed training split, one sub-folder per class.

    Returns:
        {output_index: weight} ready to pass to `model.fit(class_weight=...)`.
    """
    counts: list[int] = []
    for class_name in class_names:
        class_dir = train_dir / class_name
        n = sum(
            1
            for p in class_dir.glob("*")
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        )
        counts.append(n)

    total = sum(counts)
    n_classes = len(class_names)
    # Avoid exactly-uniform weights when data is perfectly balanced.
    weights = {
        i: (total / (n_classes * count)) if count > 0 else 1.0
        for i, count in enumerate(counts)
    }
    return weights


def make_callbacks(stage: str, report_dir: Path, checkpoint_path: Path):
    """Callbacks shared by both stages: checkpoint best-on-val, stop early.
    `stage` is only used to tag logs, so the history file tells the two
    runs apart."""
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            mode="max",
            patience=PATIENCE_HEAD if stage == "head" else PATIENCE_FT,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(str(report_dir / f"train_log_{stage}.csv"), append=False),
    ]


def merge_histories(h1, h2) -> dict[str, list[float]]:
    """Merge two Keras History objects into one flat dict (merged epoch axis)."""
    merged: dict[str, list[float]] = {}
    for key in h1.history:
        merged[key] = list(h1.history[key]) + list(h2.history.get(key, []))
    return merged


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--train", type=Path, default=REPO_ROOT / "data" / "processed" / "train")
    parser.add_argument("--val", type=Path, default=REPO_ROOT / "data" / "processed" / "val")
    parser.add_argument("--report-dir", type=Path, default=DEFAULT_REPORT_DIR)
    parser.add_argument("--out-model", type=Path, default=DEFAULT_REPORT_DIR / "best_model.keras")
    parser.add_argument("--epochs-head", type=int, default=5)
    parser.add_argument("--epochs-ft", type=int, default=15)
    parser.add_argument("--batch", type=int, default=32)
    args = parser.parse_args()

    args.report_dir.mkdir(parents=True, exist_ok=True)

    # ---- 1. Data ----------------------------------------------------------
    train_ds, val_ds, class_names = build_datasets(
        str(args.train), str(args.val), batch_size=args.batch, seed=42
    )
    # ---- 1b. Label sanity check -----------------------------------------
    # Keras labels the classes in ALPHABETICAL FOLDER order, and the trained
    # model's output index follows that order. classes.py groups by crop, so
    # the two orders legitimately differ. What must NOT happen is a folder
    # that is not a known class (silent extra label) or a labels.json written
    # in a different order than training used.
    #   -> we verify the folder SET here, and record the exact order that
    #      training used in ml/reports/class_order.json so export_onnx.py can
    #      replay it when it writes backend/app/data/labels.json.
    missing = missing_from_dataset(class_names)
    if missing:
        print(
            "[train] NOTE: the dataset has no images for "
            f"{len(missing)} canonical class(es): {missing}\n"
            f"[train] Training will cover {len(class_names)} classes only. "
            "Add data for the missing ones later and re-run prepare_data.py."
        )
    class_order_path = args.report_dir / "class_order.json"
    class_order_path.write_text(
        json.dumps(
            {
                "class_names": class_names,
                "n_classes": len(class_names),
                "source": "Keras image_dataset_from_directory (alphabetical folder order)",
                "note": "The model's output index IS this list order. export_onnx.py reads this file.",
            },
            indent=2,
        )
    )
    print(f"[train] label order ({len(class_names)} classes) saved to {class_order_path}")

    class_weights = compute_class_weights(class_names, args.train)
    print("[train] class weights (top/low):",
          sorted(class_weights.items(), key=lambda kv: -kv[1])[:3],
          "...", sorted(class_weights.items(), key=lambda kv: kv[1])[:3])

    # ---- 2. Stage 1: train the frozen model -------------------------------
    # Output width MUST match the number of classes the dataset actually has,
    # otherwise the head produces dead logits for missing classes and softmax
    # probabilities are spread over labels the model can never learn.
    model = build_model(len(class_names))
    print("[train] STAGE 1 — head training (backbone frozen)")
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=args.epochs_head,
        class_weight=class_weights,
        callbacks=make_callbacks("head", args.report_dir, args.out_model),
    )

    # ---- 3. Stage 2: fine-tune the top backbone layers --------------------
    model = unfreeze_for_finetune(model)
    print(f"[train] STAGE 2 — fine-tuning top layers at lr={STAGE2_LR}")
    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=args.epochs_ft,
        class_weight=class_weights,
        callbacks=make_callbacks("finetune", args.report_dir, args.out_model),
    )

    # ---- 4. Save merged history + final model location --------------------
    merged = merge_histories(h1, h2)
    best_val_acc = max(merged.get("val_accuracy", [0.0]))
    history_payload = {
        "class_names": class_names,   # real order used by the trained model
        "class_weights": class_weights,
        "stage1_epochs_run": len(h1.history["loss"]),
        "stage2_epochs_run": len(h2.history["loss"]),
        "stage1_lr": STAGE1_LR,
        "stage2_lr": STAGE2_LR,
        "best_val_accuracy": best_val_acc,
        "best_model_path": str(args.out_model),
        "history": merged,
    }
    history_path = args.report_dir / "history.json"
    history_path.write_text(json.dumps(history_payload, indent=2))
    print(f"[train] history written to {history_path}")
    print(f"[train] best val accuracy: {best_val_acc:.4f} (REAL number — do not edit)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/src/export_onnx.py
"""
export_onnx.py — export the trained Keras model to ONNX + parity check.

WHAT IT DOES
    1. Loads the best .keras model produced by train.py.
    2. Converts it to ONNX (tf2onnx graph conversion, opset 13).
    3. PARITY CHECK: feeds the same N random preprocessed batches through
       both the Keras model and onnxruntime and asserts outputs agree to
       within a small tolerance. This catches the classic "export changed
       the model" bug that later becomes a silent wrong-diagnosis issue.
    4. Writes:
         backend/app/data/model.onnx  (commit if < 50 MB)
         backend/app/data/labels.json (index -> {class, en, bn}; generated
                                        from ml/src/classes.py so they can
                                        never disagree).

USAGE
    python ml/src/export_onnx.py --model ml/reports/best_model.keras
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

from classes import BN_NAMES, CLASSES, EN_NAMES, LABEL_TO_INDEX, crop_of

REPO_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_MODEL = REPO_ROOT / "ml" / "reports" / "best_model.keras"
# Written by train.py: the exact output order of the trained model (Keras uses
# the alphabetical folder order, which is NOT classes.py order).
DEFAULT_CLASS_ORDER = REPO_ROOT / "ml" / "reports" / "class_order.json"
DEFAULT_OUT_DIR = REPO_ROOT / "backend" / "app" / "data"
OPSET = 13          # stable, supported by onnxruntime and tf2onnx alike
PARITY_TOLERANCE = 1e-3   # max abs diff allowed between keras and onnx outputs


def export_onnx(model, onnx_path: Path) -> None:
    """Convert the Keras model graph to an ONNX file."""
    import tensorflow as tf
    import tf2onnx

    spec = (tf.TensorSpec((None, 224, 224, 3), tf.float32, name="leaf_image_input"),)
    model_proto, _ = tf2onnx.convert.from_keras(
        model, input_signature=spec, opset=OPSET, output_path=str(onnx_path)
    )
    print(f"[export] ONNX graph written to {onnx_path} ({onnx_path.stat().st_size/1024/1024:.1f} MB)")


def run_parity_check(model, onnx_path: Path, n_batches: int = 5) -> float:
    """Compare outputs of the Keras model vs onnxruntime on random inputs.

    Returns the max absolute disagreement observed; raises SystemExit when
    it exceeds PARITY_TOLERANCE. Called BEFORE the export is declared good.
    """
    import onnxruntime as ort

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    onnx_input_name = session.get_inputs()[0].name

    max_diff = 0.0
    rng = np.random.default_rng(42)
    for _ in range(n_batches):
        # Input space: [-1, +1] floats, exactly the serving preprocessing.
        batch = rng.uniform(-1.0, 1.0, size=(4, 224, 224, 3)).astype(np.float32)
        keras_out = model.predict_on_batch(batch).numpy()
        onnx_out = session.run(None, {onnx_input_name: batch})[0]
        max_diff = max(max_diff, float(np.abs(keras_out - onnx_out).max()))

    if max_diff > PARITY_TOLERANCE:
        raise SystemExit(
            f"PARITY CHECK FAILED: max diff {max_diff:.6f} > {PARITY_TOLERANCE}\n"
            "The ONNX file does NOT reproduce the Keras model — do not ship it."
        )
    return max_diff


def load_class_order(path: Path) -> list[str]:
    """Load the exact label order the model was trained with.

    Why this exists: Keras numbers classes by ALPHABETICAL folder name, so the
    model's output index follows the dataset order (e.g. potato first), while
    classes.py is grouped by crop (rice first). Writing labels.json in
    classes.py order would map every prediction to the wrong disease — the
    one bug users could never notice until a farmer sprays the wrong field.
    train.py records the real order in ml/reports/class_order.json.

    Raises:
        SystemExit: if the file is missing or contains unknown class names.
    """
    if not path.exists():
        raise SystemExit(
            f"Class order file not found: {path}\n"
            "Run ml/src/train.py first — it writes class_order.json next to the model."
        )
    order = json.loads(path.read_text())["class_names"]
    unknown = [c for c in order if c not in LABEL_TO_INDEX]
    if unknown:
        raise SystemExit(
            f"class_order.json contains unknown classes: {unknown}\n"
            f"Known classes: {sorted(CLASSES)}"
        )
    return order


def write_labels_json(out_dir: Path, class_names: list[str]) -> None:
    """Write labels.json in the MODEL's real output order (index, en, bn)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    labels = {
        str(index): {
            "class": name,
            "en": EN_NAMES[name],
            "bn": BN_NAMES[name],
            "crop": crop_of(name),
        }
        for index, name in enumerate(class_names)
    }
    payload = {
        "n_classes": len(class_names),
        "input_size": [224, 224],
        "labels": labels,
        "note": (
            "Generated by ml/src/export_onnx.py in the model's trained output "
            "order (ml/reports/class_order.json); names from ml/src/classes.py."
        ),
    }
    path = out_dir / "labels.json"
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    print(f"[export] labels.json written to {path}")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", type=Path, default=DEFAULT_MODEL)
    parser.add_argument("--out-dir", type=Path, default=DEFAULT_OUT_DIR)
    parser.add_argument(
        "--class-order",
        type=Path,
        default=DEFAULT_CLASS_ORDER,
        help="class_order.json written by train.py (model output order)",
    )
    args = parser.parse_args()

    import tensorflow as tf  # deferred: fast --help

    if not args.model.exists():
        raise SystemExit(f"Model not found: {args.model} — train first (ml/src/train.py).")

    class_names = load_class_order(args.class_order)
    print(f"[export] model output order ({len(class_names)} classes): {class_names}")

    model = tf.keras.models.load_model(args.model)
    if model.output_shape[-1] != len(class_names):
        raise SystemExit(
            f"Model has {model.output_shape[-1]} outputs but class_order.json lists "
            f"{len(class_names)} classes — refusing to write labels.json."
        )

    onnx_path = args.out_dir / "model.onnx"

    export_onnx(model, onnx_path)
    max_diff = run_parity_check(model, onnx_path)
    write_labels_json(args.out_dir, class_names)

    print(f"[export] parity check PASSED (max diff {max_diff:.6f})")
    size_mb = onnx_path.stat().st_size / 1024 / 1024
    if size_mb > 50:
        print(f"[export] NOTE: {size_mb:.1f} MB > 50 MB — do NOT commit this file; "
              "document a download link instead (.gitignore covers *.onnx handling).")
    else:
        print(f"[export] {size_mb:.1f} MB — small enough to commit to the repo.")


if __name__ == "__main__":
    main()


## 2. Bring the prepared split from Google Drive (~10 min)

The 61-class split (train/val/test, 97,480 images) was already built locally,
so there is NO Kaggle download and NO `prepare_data.py` step on Colab.

**One-time, on your PC:** upload `New folder (2)/processed_61cls.7z` (the file
created by the repo work) to Google Drive, inside a folder named `CropDet`.
https://drive.google.com — just drag & drop the file into the `CropDet` folder.


In [ ]:
# 2a. Mount Google Drive and copy the archive into the Colab VM.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/CropDet/processed_61cls.7z'
assert os.path.exists(src), (
    'Upload New folder (2)/processed_61cls.7z to Drive, folder CropDet, then re-run.')
print('Copying ~17 GB from Drive to the VM (~2-4 min)...')
shutil.copy(src, '/content/processed_61cls.7z')
print('done')


In [ ]:
# 2b. Extract into data/processed (7z is pre-installed on Colab).
#     This lands ~17 GB in the VM disk — Pro runtimes have room; T4 default
#     (non-Pro) 50 GB disks also fit, but Pro is safer.
!apt-get install -y -qq p7zip-full >/dev/null 2>&1 || true
!7z x /content/processed_61cls.7z -o/content -y -bd
!rm /content/processed_61cls.7z   # free VM disk space
!ls data/processed

In [ ]:
# 2c. Verify the split: 61 classes, none empty.
from pathlib import Path
counts = {p.name: len(list(p.glob('*'))) for p in sorted(Path('data/processed/train').iterdir()) if p.is_dir()}
print('train classes:', len(counts))
empty = {c: n for c, n in counts.items() if n == 0}
assert len(counts) == 61 and not empty, f'UNEXPECTED: {len(counts)} classes, empty: {empty}'
print('OK: 61 classes, all non-empty')


**If any printed folder name is NOT mapped in `ml/src/prepare_data.py` ->
`ALIAS_MAP`, fix the alias (add the exact folder name) and re-run only the
next cell of section 3 — do NOT train yet.**

## 3. Sanity statistics

The split was built locally with seed 42, so prepare_data.py must NOT run here
(that would break the local/Colab split correspondence). Just look at the chart.


In [ ]:
!python ml/src/dataset_stats.py   # -> ml/reports/class_distribution.png
from IPython.display import Image, display
display(Image('ml/reports/class_distribution.png'))
print('''Checks before training:
* every class >0 images in the TOTAL column,
* no '!' zero-classes row at the bottom.
If anything is 0 -> STOP and tell the repo owner instead of training.''')

**Check the chart: every class must have >0 images in the total column.
If a rice class is missing images, the notebook must be stopped — tell the
repo owner instead of training on a partial class list.**

## 4. Training (the long cell — 1–2 h on T4 GPU)

In [ ]:
# 4a. Two-stage training. Writes:
#      ml/reports/best_model.keras   (best checkpoint by val_accuracy)
#      ml/reports/history.json       (full merged history - REAL numbers)
#      ml/reports/class_order.json   (label order for labels.json)
# batch 64 fits L4/A100; use --batch 32 if you fell back to a T4 runtime.
!python ml/src/train.py --epochs-head 5 --epochs-ft 15 --batch 64


In [ ]:
# 4b. Plot the merged history curves (for the final report).
import json, matplotlib.pyplot as plt
h = json.load(open('ml/reports/history.json'))['history']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['accuracy'], label='train'); ax[0].plot(h['val_accuracy'], label='val')
ax[0].set_title('accuracy'); ax[0].legend()
ax[1].plot(h['loss'], label='train'); ax[1].plot(h['val_loss'], label='val')
ax[1].set_title('loss'); ax[1].legend()
plt.savefig('ml/reports/history_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Evaluation + threshold calibration + ONNX export

In [ ]:
# 5a. Per-class precision/recall/F1 + confusion matrix -> ml/reports/metrics.md
!python ml/src/evaluate.py --model ml/reports/best_model.keras --data data/processed/test
# 5b. Confidence threshold sweep on validation -> ml/reports/threshold.json
!python ml/src/calibrate_threshold.py --model ml/reports/best_model.keras --val data/processed/val
!cat ml/reports/threshold.json | head -20

In [ ]:
# 5c. Export to ONNX (+ parity check) -> backend/app/data/model.onnx + labels.json
!python ml/src/export_onnx.py --model ml/reports/best_model.keras
!ls -la backend/app/data/

## 6. Collect + download the artifacts to send back

In [ ]:
# 6. Zip everything the repo owner needs, then trigger a browser download.
!cd ml/reports && cp ../../backend/app/data/labels.json . && \
  zip -q artifacts.zip metrics.md field_test.md confusion_matrix.png \
      field_confusion_matrix.png history.json history_curves.png \
      threshold.json class_order.json class_distribution.png train_log_head.csv \
      train_log_finetune.csv labels.json 2>/dev/null; true
  # field_test.* may not exist yet (need real phone photos) — that is OK.
!ls -la ml/reports/artifacts.zip
from google.colab import files
files.download('ml/reports/artifacts.zip')
# ALSO download the model file separately (not inside the zip):
files.download('backend/app/data/model.onnx')

**Send `artifacts.zip` + `model.onnx` back to the repo owner.** The metrics,
threshold and labels land in git; `model.onnx` is committed only if it is
under 50 MB.